In [1]:
"""
ML + Deep Learning Stacking/Blending Ensemble — SFS Loop  [v2]
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Upgrades over v1
────────────────
  [1] Config dataclass        — all hyper-params & paths in one place
  [2] seed_everything()       — NumPy + Python + PyTorch reproducibility
  [3] Checkpoint / resume     — reads existing CSV rows, skips done steps
  [4] OOF-based Youden thr.   — uses CV probs, not in-sample → no leakage
  [5] Stacking meta-LR        — nested cross_val_predict replaces fixed
                                 weights; learnable, still fully OOF-valid
  [6] Early stopping XGB/LGBM — split-before-SMOTE val set (clean early stop)
  [7] AMP + grad clipping MLP — faster on GPU, stable on CPU
  [8] Per-model AUC columns   — AUC_XGB, AUC_LGBM, AUC_MLP for diagnostics
  [9] Memory cleanup           — del models after each SFS step
 [10] tqdm progress bar        — live CV/Test AUC + threshold display
 [11] XGBoost deprecation fix  — use_label_encoder removed
 [12] os.makedirs safety       — auto-creates output dir
"""
from __future__ import annotations

import logging
import os
import random
import warnings
from dataclasses import dataclass

import mygene
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
)
from sklearn.model_selection import (
    StratifiedKFold, cross_val_predict, train_test_split,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)



In [2]:

# ══════════════════════════════════════════════════════════
# [1] CONFIG
# ══════════════════════════════════════════════════════════
@dataclass
class Config:
    seed: int = 42
    n_splits: int = 5
    sfs_start: int = 10        # inclusive; SFS begins at feature index 10
    n_features: int = 208      # total features in ranked pool
    smote_k: int = 3

    # ── XGBoost ──────────────────────────────────────────
    xgb_n_est: int = 300
    xgb_depth: int = 4
    xgb_lr: float = 0.05
    xgb_early_stop: int = 20   # rounds without improvement

    # ── LightGBM ─────────────────────────────────────────
    lgbm_n_est: int = 300
    lgbm_depth: int = 4
    lgbm_lr: float = 0.05
    lgbm_early_stop: int = 20

    # ── MLP ───────────────────────────────────────────────
    mlp_hidden: int = 256
    mlp_dropout: float = 0.4
    mlp_epochs: int = 60
    mlp_lr: float = 1e-3
    mlp_batch: int = 64
    mlp_clip: float = 1.0      # gradient norm clip

    # ── Ensemble mode ─────────────────────────────────────
    use_stacking: bool = True  # True → meta-LR; False → fixed weights
    w_xgb: float = 0.35        # only used when use_stacking=False
    w_lgbm: float = 0.35
    w_mlp: float = 0.30

    # ── Paths ─────────────────────────────────────────────
    data_dir: str = (
        r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026"
        r"\Gene Expression Omnibus (GEO) Dataset GSE68086"
        r"\python_desque2_lung\Binary\data\ranked rf"
    )
    out_dir: str = (
        r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026"
        r"\Gene Expression Omnibus (GEO) Dataset GSE68086"
        r"\python_desque2_lung\Binary\data\sfs"
    )
    train_csv: str = "MLDLBlend_v2_train.csv"
    test_csv: str  = "MLDLBlend_v2_test.csv"


CFG = Config()


In [3]:


# ══════════════════════════════════════════════════════════
# [2] UTILITIES
# ══════════════════════════════════════════════════════════
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def setup_logging() -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%H:%M:%S",
    )
    return logging.getLogger(__name__)


seed_everything(CFG.seed)
log = setup_logging()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Device: {device} | Stacking meta-LR: {CFG.use_stacking}")
os.makedirs(CFG.out_dir, exist_ok=True)  # [12]



00:03:30 [INFO] Device: cpu | Stacking meta-LR: True


In [4]:

# ══════════════════════════════════════════════════════════
# LOAD DATA
# ══════════════════════════════════════════════════════════
X_train = pd.read_csv(os.path.join(CFG.data_dir, "X_train_rF_DEG_train_lung.csv"), low_memory=False)
X_test  = pd.read_csv(os.path.join(CFG.data_dir, "X_test_rF_DEG_train_lung.csv"),  low_memory=False)
y_train = pd.read_csv(os.path.join(CFG.data_dir, "y_train_rF_DEG_train.csv"), low_memory=False)
y_test  = pd.read_csv(os.path.join(CFG.data_dir, "y_test_rF_DEG_train.csv"),  low_memory=False)

# ── Ensembl → gene symbol ─────────────────────────────────
ensembl_ids = [c for c in X_train.columns if c.startswith("ENSG")]
log.info(f"Querying MyGene.info for {len(ensembl_ids)} Ensembl IDs…")
mg      = mygene.MyGeneInfo()
results = mg.querymany(ensembl_ids, scopes="ensembl.gene", fields="symbol", species="human")
id_map  = {r["query"]: r.get("symbol", r["query"]) for r in results}
X_train.rename(columns=id_map, inplace=True)
X_test.rename(columns=id_map, inplace=True)
X_train = X_train.iloc[:, :CFG.n_features]
X_test  = X_test.iloc[:, :CFG.n_features]
log.info("Gene symbol conversion complete.")

# ── Scale ─────────────────────────────────────────────────
scaler     = StandardScaler()
x_train_sc = scaler.fit_transform(X_train)
x_test_sc  = scaler.transform(X_test)

list_feature = list(X_train.columns)
ranks = [[list_feature[j] for j in range(i + 1)] for i in range(len(list_feature))]
ratio = float((y_train["cancer_type"] == 0).sum() / (y_train["cancer_type"] == 1).sum())
skf   = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)


00:03:30 [INFO] Querying MyGene.info for 211 Ensembl IDs…
00:03:30 [INFO] querying 1-211 ...
00:03:31 [INFO] HTTP Request: POST https://mygene.info/v3/query/ "HTTP/1.1 200 OK"
00:03:32 [INFO] Finished.
00:03:32 [WARNING] 5 input query terms found no hit:	['ENSG00000197459', 'ENSG00000267243', 'ENSG00000136653', 'ENSG00000163386', 'ENSG00000148362']
00:03:32 [INFO] Pass "returnall=True" to return complete lists of duplicate or missing query terms.
00:03:32 [INFO] Gene symbol conversion complete.


In [5]:


# ══════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════
def cfm(y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> np.ndarray:
    auc = roc_auc_score(y_true, y_prob)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    acc  = accuracy_score(y_true, y_pred)
    rec  = tp / (tp + fn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    spec = tn / (tn + fp)
    f1   = (2 * rec * prec) / (rec + prec) if (rec + prec) > 0 else 0.0
    # returns [acc, recall, precision, specificity, sensitivity, f1, auc]
    return np.array([acc, rec, prec, spec, rec, f1, auc])



In [6]:

# ══════════════════════════════════════════════════════════
# [7] MLP — Residual architecture, AMP + gradient clipping
# ══════════════════════════════════════════════════════════
class ResidualMLP(nn.Module):
    def __init__(self, in_f: int, hidden: int = 256, dropout: float = 0.4):
        super().__init__()
        h = hidden
        self.b1   = nn.Sequential(
            nn.Linear(in_f, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h, h),    nn.BatchNorm1d(h),
        )
        self.sk1  = nn.Linear(in_f, h)
        self.b2   = nn.Sequential(
            nn.Linear(h, h // 2), nn.BatchNorm1d(h // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h // 2, h // 2), nn.BatchNorm1d(h // 2),
        )
        self.sk2  = nn.Linear(h, h // 2)
        self.head = nn.Linear(h // 2, 1)
        self.act  = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.act(self.b1(x) + self.sk1(x))
        x = self.act(self.b2(x) + self.sk2(x))
        return self.head(x)


def train_mlp(X_tr: np.ndarray, y_tr: np.ndarray,
              n_feat: int, dev: torch.device) -> ResidualMLP:
    pos_w  = torch.tensor(
        [(y_tr == 0).sum() / max((y_tr == 1).sum(), 1)], dtype=torch.float32
    ).to(dev)
    X_t    = torch.tensor(X_tr, dtype=torch.float32).to(dev)
    y_t    = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1).to(dev)
    loader = DataLoader(
        TensorDataset(X_t, y_t), batch_size=CFG.mlp_batch, shuffle=True, drop_last=False
    )
    model      = ResidualMLP(n_feat, CFG.mlp_hidden, CFG.mlp_dropout).to(dev)
    crit       = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt        = torch.optim.AdamW(model.parameters(), lr=CFG.mlp_lr, weight_decay=1e-4)
    sched      = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.mlp_epochs)
    use_amp    = dev.type == "cuda"
    amp_scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    model.train()
    for _ in range(CFG.mlp_epochs):
        for xb, yb in loader:
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = crit(model(xb), yb)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.mlp_clip)  # [7]
            amp_scaler.step(opt)
            amp_scaler.update()
        sched.step()
    return model


@torch.no_grad()  # [7]
def predict_mlp(model: ResidualMLP, X: np.ndarray,
                dev: torch.device, bs: int = 256) -> np.ndarray:
    model.eval()
    X_t   = torch.tensor(X, dtype=torch.float32).to(dev)
    probs = []
    for s in range(0, len(X_t), bs):
        probs.append(torch.sigmoid(model(X_t[s:s + bs])).cpu().numpy())
    return np.vstack(probs).flatten()



In [7]:

# ══════════════════════════════════════════════════════════
# [6] TREE FACTORIES — split-before-SMOTE for clean early stopping
# ══════════════════════════════════════════════════════════
def _smote_split(X: np.ndarray, y: np.ndarray, val_frac: float = 0.15):
    """
    1. Split original data → train_s / val (no synthetic samples in val)
    2. Apply SMOTE only to train_s
    Returns: X_sm, y_sm, X_val, y_val
    """
    X_s, X_v, y_s, y_v = train_test_split(
        X, y, test_size=val_frac, stratify=y, random_state=CFG.seed
    )
    X_sm, y_sm = SMOTE(k_neighbors=CFG.smote_k, random_state=CFG.seed).fit_resample(X_s, y_s)
    return X_sm, y_sm, X_v, y_v


def fit_xgb(X: np.ndarray, y: np.ndarray) -> XGBClassifier:  # [11] no use_label_encoder
    X_sm, y_sm, X_v, y_v = _smote_split(X, y)
    m = XGBClassifier(
        n_estimators=CFG.xgb_n_est, max_depth=CFG.xgb_depth, learning_rate=CFG.xgb_lr,
        scale_pos_weight=ratio, eval_metric="logloss",
        early_stopping_rounds=CFG.xgb_early_stop,
        random_state=CFG.seed, n_jobs=-1,
    )
    m.fit(X_sm, y_sm, eval_set=[(X_v, y_v)], verbose=False)
    return m


def fit_lgbm(X: np.ndarray, y: np.ndarray) -> LGBMClassifier:
    X_sm, y_sm, X_v, y_v = _smote_split(X, y)
    m = LGBMClassifier(
        n_estimators=CFG.lgbm_n_est, max_depth=CFG.lgbm_depth, learning_rate=CFG.lgbm_lr,
        is_unbalance=True, random_state=CFG.seed, n_jobs=-1, verbose=-1,
    )
    m.fit(
        X_sm, y_sm, eval_set=[(X_v, y_v)],
        callbacks=[
            lgb.early_stopping(CFG.lgbm_early_stop, verbose=False),
            lgb.log_evaluation(-1),
        ],
    )
    return m



In [8]:

# ══════════════════════════════════════════════════════════
# [3] CHECKPOINT / RESUME
# ══════════════════════════════════════════════════════════
COLS = [
    "Gene", "ACC", "Recall", "Precision", "Sp", "Sn", "f1_Score", "AUC",
    "AUC_XGB", "AUC_LGBM", "AUC_MLP",
]
train_out = os.path.join(CFG.out_dir, CFG.train_csv)
test_out  = os.path.join(CFG.out_dir, CFG.test_csv)

result_train: list = []
result_test:  list = []
resume_from = CFG.sfs_start

if os.path.exists(train_out):
    _ck = pd.read_csv(train_out)
    if len(_ck) > 0:
        result_train = _ck.values.tolist()
        resume_from  = CFG.sfs_start + len(_ck)
        log.info(f"Resuming from feature index {resume_from} ({len(_ck)} steps already done).")

if os.path.exists(test_out):
    _ck2 = pd.read_csv(test_out)
    if len(_ck2) > 0:
        result_test = _ck2.values.tolist()



In [9]:

# ══════════════════════════════════════════════════════════
# [4][5][8][9][10] SFS LOOP
# ══════════════════════════════════════════════════════════
log.info("Starting ML+DL Blending/Stacking SFS loop…")
n_steps = len(ranks) - CFG.sfs_start

with tqdm(total=n_steps, desc="SFS", unit="step",
          initial=resume_from - CFG.sfs_start) as pbar:

    for i in range(CFG.sfs_start, len(ranks)):
        if i < resume_from:
            continue

        n_feat = i + 1
        X_tr   = x_train_sc[:, :n_feat]
        X_te   = x_test_sc[:, :n_feat]
        y_tr   = y_train["cancer_type"].values
        y_te   = y_test["cancer_type"].values

        # ── [4] Collect OOF predictions per model (CV) ───────
        oof_xgb  = np.zeros(len(y_tr))
        oof_lgbm = np.zeros(len(y_tr))
        oof_mlp  = np.zeros(len(y_tr))

        for tr_idx, val_idx in skf.split(X_tr, y_tr):
            Xf_tr, Xf_val = X_tr[tr_idx], X_tr[val_idx]
            yf_tr         = y_tr[tr_idx]

            xgb_f  = fit_xgb(Xf_tr, yf_tr)
            lgbm_f = fit_lgbm(Xf_tr, yf_tr)
            mlp_f  = train_mlp(Xf_tr, yf_tr, n_feat, device)

            oof_xgb[val_idx]  = xgb_f.predict_proba(Xf_val)[:, 1]
            oof_lgbm[val_idx] = lgbm_f.predict_proba(Xf_val)[:, 1]
            oof_mlp[val_idx]  = predict_mlp(mlp_f, Xf_val, device)
            del xgb_f, lgbm_f, mlp_f  # [9]

        # ── [5] Blend / stack OOF probabilities ──────────────
        if CFG.use_stacking:
            oof_stack = np.column_stack([oof_xgb, oof_lgbm, oof_mlp])
            meta_skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
            meta      = LogisticRegression(C=1.0, max_iter=2000, random_state=CFG.seed)
            # Nested OOF → fully unbiased CV metric for the meta-layer
            cv_probs  = cross_val_predict(
                meta, oof_stack, y_tr, cv=meta_skf, method="predict_proba"
            )[:, 1]
            meta.fit(oof_stack, y_tr)   # refit on all OOF for test-time inference
        else:
            cv_probs = (
                CFG.w_xgb  * oof_xgb
                + CFG.w_lgbm * oof_lgbm
                + CFG.w_mlp  * oof_mlp
            )

        # ── [4] OOF-based Youden threshold (no leakage) ──────
        fpr_cv, tpr_cv, thr_cv = roc_curve(y_tr, cv_probs)
        best_thr = float(thr_cv[np.argmax(tpr_cv - fpr_cv)])

        # ── CV metrics ───────────────────────────────────────
        y_cv_pred   = (cv_probs >= 0.5).astype(int)
        cv_m        = cfm(y_tr, y_cv_pred, cv_probs)
        auc_xgb_cv  = roc_auc_score(y_tr, oof_xgb)
        auc_lgbm_cv = roc_auc_score(y_tr, oof_lgbm)
        auc_mlp_cv  = roc_auc_score(y_tr, oof_mlp)

        # ── Fit final models on full training data ────────────
        xgb_final  = fit_xgb(X_tr, y_tr)
        lgbm_final = fit_lgbm(X_tr, y_tr)
        mlp_final  = train_mlp(X_tr, y_tr, n_feat, device)

        # ── Test prediction with locked Youden threshold ──────
        p_xgb_te  = xgb_final.predict_proba(X_te)[:, 1]
        p_lgbm_te = lgbm_final.predict_proba(X_te)[:, 1]
        p_mlp_te  = predict_mlp(mlp_final, X_te, device)

        if CFG.use_stacking:
            p_blend_te = meta.predict_proba(
                np.column_stack([p_xgb_te, p_lgbm_te, p_mlp_te])
            )[:, 1]
        else:
            p_blend_te = (
                CFG.w_xgb  * p_xgb_te
                + CFG.w_lgbm * p_lgbm_te
                + CFG.w_mlp  * p_mlp_te
            )

        y_pred_te = (p_blend_te >= best_thr).astype(int)
        te_m      = cfm(y_te, y_pred_te, p_blend_te)

        del xgb_final, lgbm_final, mlp_final  # [9]

        # ── [8] Save with per-model AUC columns ──────────────
        result_train.append([
            ranks[i],
            *[f"{v * 100:.2f}%" for v in cv_m],
            f"{auc_xgb_cv  * 100:.2f}%",
            f"{auc_lgbm_cv * 100:.2f}%",
            f"{auc_mlp_cv  * 100:.2f}%",
        ])
        pd.DataFrame(result_train, columns=COLS).to_csv(train_out, index=False)

        result_test.append([
            ranks[i],
            *[f"{v * 100:.2f}%" for v in te_m],
            f"{roc_auc_score(y_te, p_xgb_te)  * 100:.2f}%",
            f"{roc_auc_score(y_te, p_lgbm_te) * 100:.2f}%",
            f"{roc_auc_score(y_te, p_mlp_te)  * 100:.2f}%",
        ])
        pd.DataFrame(result_test, columns=COLS).to_csv(test_out, index=False)

        # ── [10] tqdm postfix ─────────────────────────────────
        pbar.set_postfix(
            n=n_feat,
            CV_AUC=f"{cv_m[6] * 100:.1f}%",
            Te_AUC=f"{te_m[6] * 100:.1f}%",
            thr=f"{best_thr:.3f}",
        )
        pbar.update(1)

log.info("Done! Results saved.")
df_tr = pd.read_csv(train_out)
df_te = pd.read_csv(test_out)
print("\nFinal 5 training rows:\n", df_tr.tail(5).to_string(index=False))
print("\nFinal 5 test rows:\n",     df_te.tail(5).to_string(index=False))

00:03:32 [INFO] Starting ML+DL Blending/Stacking SFS loop…
SFS: 100%|██████████| 198/198 [8:53:25<00:00, 161.64s/step, CV_AUC=96.3%, Te_AUC=95.3%, n=208, thr=0.489]  
08:56:57 [INFO] Done! Results saved.



Final 5 training rows:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                